### 1. Import thư viện & Khởi tạo cấu hình

In [ ]:
import os
import re
import string
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Tắt cảnh báo không cần thiết để báo cáo sạch hơn
# warnings.filterwarnings('ignore')

# Khai báo các hằng số (Constants) dùng chung cho toàn dự án
RANDOM_STATE = 42
MAX_TFIDF_FEATURES = 5000

# Download NLTK data (Chỉ cần chạy 1 lần)
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('omw-1.4')

### 2. Hàm Tiền xử lý dữ liệu (Data Preprocessing Pipeline)

In [ ]:
def load_and_merge_data(fake_path: str, true_path: str) -> pd.DataFrame:
    """
    Đọc dữ liệu từ file CSV, gán nhãn và xáo trộn dữ liệu.
    Input:
        - fake_path: Đường dẫn tới file tin giả
        - true_path: Đường dẫn tới file tin thật
    Output:
        - DataFrame đã được gộp, gán nhãn và xáo trộn ngẫu nhiên.
    """
    df_fake = pd.read_csv(fake_path)
    df_true = pd.read_csv(true_path)

    # Gán nhãn (1: Fake, 0: True)
    df_fake['label'] = 1
    df_true['label'] = 0

    # Gộp và xáo trộn (shuffle) dữ liệu để tránh thiên vị khi train
    df = pd.concat([df_fake, df_true], axis=0)
    df = df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
    
    return df

def clean_text_basic(text: str) -> str:
    """
    Làm sạch văn bản thô: Xóa nguồn tin, URL, HTML, dấu câu và khoảng trắng thừa.
    """
    text = str(text)
    
    # Xóa Nguồn tin (ví dụ: "WASHINGTON (Reuters) -")
    text = re.sub(r'^.*?\(reuters\)\s*[-–—]\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^.*?\s?[-–—]\s?', '', text, count=1)
    
    # Chuyển về chữ thường
    text = text.lower()
    
    # Xóa URL và thẻ HTML
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    
    # Xóa dấu câu (punctuation)
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Xử lý khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def apply_lemmatization(text: str, lemmatizer, stop_words: set) -> str:
    """
    Tiền xử lý NLP chuyên sâu: Tách từ, xóa Stopwords và Lemmatize (đưa từ về gốc).
    """
    words = str(text).split()
    clean_words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(clean_words)

### 3. Hàm Trích xuất Đặc trưng (Feature Engineering)

In [ ]:
def extract_meta_features(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    """
    Trích xuất các đặc trưng thống kê (Meta-features) từ văn bản gốc.
    Các đặc trưng này giúp mô hình nắm bắt phong cách viết (tin giả thường dùng nhiều dấu câu/chữ hoa).
    """
    meta_df = pd.DataFrame()
    
    # 1. Độ dài văn bản (số lượng từ)
    meta_df['body_len'] = df[text_col].apply(lambda x: len(str(x).split()))
    
    # 2. Tỷ lệ dấu câu trên tổng số từ
    def get_punct_ratio(text):
        words = str(text).split()
        if not words: return 0
        punct_count = sum(1 for char in str(text) if not char.isalnum() and char != ' ')
        return round((punct_count / len(words)) * 100, 3)
        
    meta_df['punct_per_word'] = df[text_col].apply(get_punct_ratio)
    
    # 3. Tỷ lệ chữ in hoa trên tổng số từ
    def get_cap_ratio(text):
        words = str(text).split()
        if not words: return 0
        cap_count = sum(1 for char in str(text) if char.isupper())
        return round((cap_count / len(words)) * 100, 3)
        
    meta_df['cap_per_word'] = df[text_col].apply(get_cap_ratio)
    
    return meta_df

def prepare_modeling_data(df: pd.DataFrame, text_clean_col: str, text_raw_col: str):
    """
    Tạo Pipeline chuẩn bị dữ liệu (ngăn chặn Data Leakage).
    Chia Train/Test TRƯỚC, sau đó mới Fit TF-IDF và Scaler trên tập Train.
    """
    # 1. Lấy meta-features từ văn bản CHƯA clean để đếm chính xác chữ hoa/dấu câu
    meta_features = extract_meta_features(df, text_raw_col)
    
    # 2. Chia tập Train/Test (Chia trước khi trích xuất TF-IDF)
    X_text = df[text_clean_col]
    X_meta = meta_features
    y = df['label']
    
    # Stratify=y giúp giữ nguyên tỷ lệ nhãn giữa tập Train và Test
    X_text_train, X_text_test, X_meta_train, X_meta_test, y_train, y_test = train_test_split(
        X_text, X_meta, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
    )
    
    # 3. Fit & Transform TF-IDF (Chỉ Fit trên TRAIN)
    tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=MAX_TFIDF_FEATURES)
    X_tfidf_train = tfidf.fit_transform(X_text_train).toarray()
    X_tfidf_test = tfidf.transform(X_text_test).toarray() # Chỉ transform cho Test
    
    # 4. Gộp đặc trưng TF-IDF và Meta-features
    X_train_combined = np.hstack((X_tfidf_train, X_meta_train.values))
    X_test_combined = np.hstack((X_tfidf_test, X_meta_test.values))
    
    # 5. Scale dữ liệu (Chỉ Fit trên TRAIN)
    scaler = MinMaxScaler()
    X_train_final = scaler.fit_transform(X_train_combined)
    X_test_final = scaler.transform(X_test_combined)
    
    return X_train_final, X_test_final, y_train, y_test

### 4. Hàm Đánh giá và Huấn luyện chung (Model Factory)

In [ ]:
def train_and_evaluate_model(model, X_train, X_test, y_train, y_test, model_name: str, cmap: str = 'Blues'):
    """
    Hàm tổng quát hóa việc huấn luyện, dự đoán và đánh giá bất kỳ mô hình nào.
    Áp dụng nguyên tắc DRY (Don't Repeat Yourself).
    """
    print(f"\n" + "="*50)
    print(f" ĐANG HUẤN LUYỆN MÔ HÌNH: {model_name} ".center(50, '='))
    print("="*50)
    
    # Huấn luyện mô hình
    model.fit(X_train, y_train)
    
    # Dự đoán
    y_pred = model.predict(X_test)
    
    # Đánh giá độ chính xác
    acc = accuracy_score(y_test, y_pred)
    report_dict = classification_report(y_test, y_pred, output_dict=True)
    
    print(f"Accuracy Score: {acc:.4f}\n")
    print("Classification Report:")
    print(classification_report(y_test, y_pred))
    
    # Vẽ Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, 
                xticklabels=['Tin thật (0)', 'Tin giả (1)'], 
                yticklabels=['Tin thật (0)', 'Tin giả (1)'])
    plt.title(f'Confusion Matrix - {model_name}')
    plt.ylabel('Thực tế (Actual)')
    plt.xlabel('Dự đoán (Predicted)')
    plt.show()
    
    return acc, report_dict

### 5. Thực thi luồng chính

In [ ]:
# 1. Khởi tạo danh sách Stopwords
stop_words = set(stopwords.words('english'))
political_stopwords = {'said', 'also', 'would', 'could', 'told', 'reuters', 'washington', 'statement', 'press'}
stop_words.update(political_stopwords)
lemmatizer = WordNetLemmatizer()

# 2. Xử lý Pipeline
print("1. Đang load dữ liệu...")
df = load_and_merge_data('../data/raw/Fake.csv', '../data/raw/True.csv')

print("2. Đang làm sạch dữ liệu cơ bản...")
# Tạo 2 cột text để so sánh
df['text_only'] = df['text'].apply(clean_text_basic)
df['title_text'] = (df['title'] + " " + df['text']).apply(clean_text_basic)

print("3. Đang áp dụng Lemmatization (Có thể mất vài phút)...")
df['text_only_clean'] = df['text_only'].apply(lambda x: apply_lemmatization(x, lemmatizer, stop_words))
df['title_text_clean'] = df['title_text'].apply(lambda x: apply_lemmatization(x, lemmatizer, stop_words))

print("4. Đang trích xuất đặc trưng và chia tập dữ liệu...")
# Lưu ý: Cột gốc chưa làm sạch (text/title) dùng để đếm đặc trưng Meta chính xác nhất
X_train_text, X_test_text, y_train, y_test = prepare_modeling_data(df, 'text_only_clean', 'text')
X_train_title, X_test_title, _, _ = prepare_modeling_data(df, 'title_text_clean', 'title')

# 5. Huấn luyện mô hình Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
acc_lr_text, rep_lr_text = train_and_evaluate_model(lr_model, X_train_text, X_test_text, y_train, y_test, "Logistic Regression (Text Only)", "Blues")
acc_lr_title, rep_lr_title = train_and_evaluate_model(lr_model, X_train_title, X_test_title, y_train, y_test, "Logistic Regression (Title + Text)", "Blues")

# 6. Huấn luyện mô hình Naive Bayes
nb_model = MultinomialNB()
acc_nb_text, rep_nb_text = train_and_evaluate_model(nb_model, X_train_text, X_test_text, y_train, y_test, "Naive Bayes (Text Only)", "Greens")
acc_nb_title, rep_nb_title = train_and_evaluate_model(nb_model, X_train_title, X_test_title, y_train, y_test, "Naive Bayes (Title + Text)", "Greens")

# 7. Huấn luyện mô hình SVM
svm_model = LinearSVC(random_state=RANDOM_STATE)
acc_svm_text, rep_svm_text = train_and_evaluate_model(svm_model, X_train_text, X_test_text, y_train, y_test, "SVM (Text Only)", "Oranges")
acc_svm_title, rep_svm_title = train_and_evaluate_model(svm_model, X_train_title, X_test_title, y_train, y_test, "SVM (Title + Text)", "Oranges")

# 8. Bảng tổng kết
summary_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "SVM"],
    "Accuracy (Title + Text)": [acc_lr_title, acc_nb_title, acc_svm_title],
    "Accuracy (Text Only)": [acc_lr_text, acc_nb_text, acc_svm_text]
})

print("\nBẢNG TỔNG KẾT KẾT QUẢ THỰC NGHIỆM:")
display(summary_df)